In [ ]:
# Cell 3: The Fine-Tuning Script (Corrected for Modern TRL/Transformers)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments, 
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer 
from transformers.trainer_utils import get_last_checkpoint
import json
print(f"✅ Script is running with Process ID (PID): {os.getpid()}")

# --- 1. Configuration ---
model_path = "./models/Qwen2.5-3B-Instruct" 
dataset_path = "./data/train_data.jsonl"
new_model_name = "Qwen2.5-3B-Instruct_Address_Formatter"
MAX_LENGTH = 256

# --- 2. Load the Dataset ---
print("Loading dataset...")
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"✅ Dataset loaded with {len(dataset)} records.")

def format_as_chat(example):
    # Convert the output dict to a compact JSON string
    output_json = json.dumps(example["output"], ensure_ascii=False)
    messages = [
        {
            "role": "system",
            "content": "You are an address formatting assistant. You always return the formatted address as a JSON object with 'line1' and 'line2'."
        },
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": output_json}
    ]
    
    example["text"] = messages   # store the message list temporarily
    return example

dataset = dataset.map(format_as_chat)
# Now split into train/validation (10% for validation)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

# --- 3. Load Tokenizer & Model ---
print("Loading tokenizer and model...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map={"": 0} # single visible GPU is now index 0
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.15,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# --- Manually tokenize the dataset ---
print("Manually tokenizing and padding all samples...")
def tokenize_and_pad_function(examples):
    # Each 'text' is now a list of [messages], we apply the template to build the string
    conversations = []
    for msgs in examples["text"]:   # msgs is a list of message dicts
        conversation = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        conversations.append(conversation)
    return tokenizer(
        conversations,             
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized_train = dataset.map(
    tokenize_and_pad_function,
    batched=True,
    remove_columns=dataset.column_names
)
tokenized_eval = eval_dataset.map(
    tokenize_and_pad_function,
    batched=True,
    remove_columns=eval_dataset.column_names,
)
print("✅ Dataset tokenized and padded.")


# --- 4. Configure and Run the Trainer ---
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    # --- CHANGES ---
    num_train_epochs=3,          # VERY IMPORTANT: With a small dataset, only train for ONE epoch. More epochs just encourage memorization.
    learning_rate=2e-5,          # Use a smaller learning rate. A high rate helps it memorize faster.
    weight_decay=0.01,           # Add weight decay. This is a regularization technique that penalizes large weights, making memorization harder.
    warmup_steps=0.1,
    lr_scheduler_type="cosine",  # Use a cosine learning rate scheduler.
    # --- END OF CHANGES ---
    # fp16=False,
    bf16=True,
    logging_steps=10,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=20,
    eval_strategy="steps",
    eval_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

# We use the standard `Trainer` from the `transformers` library.
# The SFT functionality is now built-in.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)        # Set packing to False for this simple case
)

print("🚀 Starting fine-tuning...")
last_checkpoint = get_last_checkpoint(training_args.output_dir)
trainer.train(resume_from_checkpoint=last_checkpoint)
print("✅ Fine-tuning complete!")

# --- 5. Save the Trained LoRA Adapters ---
print(f"Saving the LoRA adapters to '{new_model_name}'...")
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)
print("✅ Adapters saved.")